In [30]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql.types import DateType

In [4]:
spark = SparkSession.builder.appName("Data_Cleaning").getOrCreate()

In [5]:
data_folder = "./sales_forecasting_data"

train = spark.read.csv(f"{data_folder}/train.csv", header=True)
stores = spark.read.csv(f"{data_folder}/stores.csv", header=True)
transactions = spark.read.csv(f"{data_folder}/transactions.csv", header=True)
oil = spark.read.csv(f"{data_folder}/oil.csv", header=True)
items = spark.read.csv(f"{data_folder}/items.csv", header=True)
holidays_events = spark.read.csv(f"{data_folder}/holidays_events.csv", header=True)


So, I started by looking at each of the data.

I used PySpark because the dataset was too large for Pandas, causing the kernel to crash. But sometimes I still convert the data to Pandas to utilize some of its functions.

In [7]:
train.show(5)

+---+----------+---------+--------+----------+-----------+
| id|      date|store_nbr|item_nbr|unit_sales|onpromotion|
+---+----------+---------+--------+----------+-----------+
|  0|2013-01-01|       25|  103665|       7.0|       NULL|
|  1|2013-01-01|       25|  105574|       1.0|       NULL|
|  2|2013-01-01|       25|  105575|       2.0|       NULL|
|  3|2013-01-01|       25|  108079|       1.0|       NULL|
|  4|2013-01-01|       25|  108701|       1.0|       NULL|
+---+----------+---------+--------+----------+-----------+
only showing top 5 rows


In [8]:
train = train.fillna({'onpromotion': False})

There are nulls in the onpromotion column, so I filled then with False.

In [9]:
stores.show(5)

+---------+-------------+--------------------+----+-------+
|store_nbr|         city|               state|type|cluster|
+---------+-------------+--------------------+----+-------+
|        1|        Quito|           Pichincha|   D|     13|
|        2|        Quito|           Pichincha|   D|     13|
|        3|        Quito|           Pichincha|   D|      8|
|        4|        Quito|           Pichincha|   D|      9|
|        5|Santo Domingo|Santo Domingo de ...|   D|      4|
+---------+-------------+--------------------+----+-------+
only showing top 5 rows


In [10]:
transactions.show(5)

+----------+---------+------------+
|      date|store_nbr|transactions|
+----------+---------+------------+
|2013-01-01|       25|         770|
|2013-01-02|        1|        2111|
|2013-01-02|        2|        2358|
|2013-01-02|        3|        3487|
|2013-01-02|        4|        1922|
+----------+---------+------------+
only showing top 5 rows


In [ ]:
transactions.select(
    F.sum(F.when(F.col("transactions").isNull(), 1).otherwise(0)).alias("null_count")
).show()

+----------+
|null_count|
+----------+
|         0|
+----------+



In [12]:
oil.show(5)

+----------+----------+
|      date|dcoilwtico|
+----------+----------+
|2013-01-01|      NULL|
|2013-01-02|     93.14|
|2013-01-03|     92.97|
|2013-01-04|     93.12|
|2013-01-07|      93.2|
+----------+----------+
only showing top 5 rows


There are some null value in the oil price as well, so interpolate them linearly with pandas since it does not make sense to fill them with 0.

I also adding missing dates from the oil and interpolate them as well. The first and last date is filled using forward and backward fills.

In [ ]:
# Convert to Pandas
oil_pd = oil.toPandas()
oil_pd["date"] = pd.to_datetime(oil_pd["date"])

# Create full date range and merge
full_dates = pd.DataFrame({"date": pd.date_range(oil_pd["date"].min(), oil_pd["date"].max())})
oil_pd = full_dates.merge(oil_pd, on="date", how="left")

# Interpolate missing values
oil_pd["dcoilwtico"] = oil_pd["dcoilwtico"].interpolate(method="linear").ffill().bfill()

# Convert back to Spark
oil = spark.createDataFrame(oil_pd)
oil = oil.withColumn("date", F.col("date").cast(DateType()))


/var/folders/9k/xqsk1xxs1wj_j4l1_6f_11140000gn/T/ipykernel_21970/2344505521.py:12: FutureWarning: Series.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  oil_pd["dcoilwtico"] = oil_pd["dcoilwtico"].interpolate(method="linear").ffill().bfill()


In [32]:
oil_pd.isnull().sum()

date          0
dcoilwtico    0
dtype: int64

In [33]:
oil.show(5)

+----------+----------+
|      date|dcoilwtico|
+----------+----------+
|2013-01-01|     93.14|
|2013-01-02|     93.14|
|2013-01-03|     92.97|
|2013-01-04|     93.12|
|2013-01-05|     93.12|
+----------+----------+
only showing top 5 rows


In [34]:
items.show(5)

+--------+------------+-----+----------+
|item_nbr|      family|class|perishable|
+--------+------------+-----+----------+
|   96995|   GROCERY I| 1093|         0|
|   99197|   GROCERY I| 1067|         0|
|  103501|    CLEANING| 3008|         0|
|  103520|   GROCERY I| 1028|         0|
|  103665|BREAD/BAKERY| 2712|         1|
+--------+------------+-----+----------+
only showing top 5 rows


In [35]:
holidays_events.show(5)
print(holidays_events.count())

+----------+-------+--------+-----------+--------------------+-----------+
|      date|   type|  locale|locale_name|         description|transferred|
+----------+-------+--------+-----------+--------------------+-----------+
|2012-03-02|Holiday|   Local|      Manta|  Fundacion de Manta|      False|
|2012-04-01|Holiday|Regional|   Cotopaxi|Provincializacion...|      False|
|2012-04-12|Holiday|   Local|     Cuenca| Fundacion de Cuenca|      False|
|2012-04-14|Holiday|   Local|   Libertad|Cantonizacion de ...|      False|
|2012-04-21|Holiday|   Local|   Riobamba|Cantonizacion de ...|      False|
+----------+-------+--------+-----------+--------------------+-----------+
only showing top 5 rows
338


In [36]:
# Drop transferred holiday
holidays_events = holidays_events.filter(F.col("transferred") == False)
holidays_events.show(5)

print(holidays_events.count())

+----------+-------+--------+-----------+--------------------+-----------+
|      date|   type|  locale|locale_name|         description|transferred|
+----------+-------+--------+-----------+--------------------+-----------+
|2012-03-02|Holiday|   Local|      Manta|  Fundacion de Manta|      False|
|2012-04-01|Holiday|Regional|   Cotopaxi|Provincializacion...|      False|
|2012-04-12|Holiday|   Local|     Cuenca| Fundacion de Cuenca|      False|
|2012-04-14|Holiday|   Local|   Libertad|Cantonizacion de ...|      False|
|2012-04-21|Holiday|   Local|   Riobamba|Cantonizacion de ...|      False|
+----------+-------+--------+-----------+--------------------+-----------+
only showing top 5 rows
338


In [37]:
# Merge data
df = train.join(items, on="item_nbr", how="left")
df = df.join(stores, on="store_nbr", how="left")
df = df.join(transactions, on=["date", "store_nbr"], how="left")
df = df.join(oil, on="date", how="left")
df = df.join(holidays_events.select(F.col("date"), F.col("type").alias("holiday_type")), on="date", how="left")

df.show()

+----------+---------+--------+---+----------+-----------+-------------+-----+----------+-------+-----------+----+-------+------------+----------+------------+
|      date|store_nbr|item_nbr| id|unit_sales|onpromotion|       family|class|perishable|   city|      state|type|cluster|transactions|dcoilwtico|holiday_type|
+----------+---------+--------+---+----------+-----------+-------------+-----+----------+-------+-----------+----+-------+------------+----------+------------+
|2013-01-01|       25|  103665|  0|       7.0|      false| BREAD/BAKERY| 2712|         1|Salinas|Santa Elena|   D|      1|         770|     93.14|     Holiday|
|2013-01-01|       25|  105574|  1|       1.0|      false|    GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|     Holiday|
|2013-01-01|       25|  105575|  2|       2.0|      false|    GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|     Holiday|
|2013-01-01|       25|  108079|  3|     

After cleaning null values, I joined them together into one dataframe to make it easy to do EDA.

In [38]:
df = df.withColumn("holiday", F.when(F.col("holiday_type") == "Holiday", True).otherwise(False))
df = df.drop("holiday_type")

Then, I replaced holiday_type column to a holiday flag one, indicating whether it's a holiday (True) or not (False).

In [39]:
df.show()

+----------+---------+--------+---+----------+-----------+-------------+-----+----------+-------+-----------+----+-------+------------+----------+-------+
|      date|store_nbr|item_nbr| id|unit_sales|onpromotion|       family|class|perishable|   city|      state|type|cluster|transactions|dcoilwtico|holiday|
+----------+---------+--------+---+----------+-----------+-------------+-----+----------+-------+-----------+----+-------+------------+----------+-------+
|2013-01-01|       25|  103665|  0|       7.0|      false| BREAD/BAKERY| 2712|         1|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  105574|  1|       1.0|      false|    GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  105575|  2|       2.0|      false|    GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  108079|  3|       1.0|      false|    GROCERY 

In [40]:
# Count nulls for each column
null_counts = []

for column_name in df.columns:
    null_count = F.sum(F.col(column_name).isNull().cast("int")).alias(column_name)
    null_counts.append(null_count)

df.select(null_counts).show()

+----+---------+--------+---+----------+-----------+------+-----+----------+----+-----+----+-------+------------+----------+-------+
|date|store_nbr|item_nbr| id|unit_sales|onpromotion|family|class|perishable|city|state|type|cluster|transactions|dcoilwtico|holiday|
+----+---------+--------+---+----------+-----------+------+-----+----------+----+-----+----+-------+------------+----------+-------+
|   0|        0|       0|  0|         0|          0|     0|    0|         0|   0|    0|   0|      0|      214625|         0|      0|
+----+---------+--------+---+----------+-----------+------+-----+----------+----+-----+----+-------+------------+----------+-------+



I checked again for any missing values, look like there are no transactions in some store for certain day, or some of the trainactions value in some store aren't recorded, so there are null values.

I filled them with 0 to be safe since it could be that there are no transaction.

In [41]:
# Fill missing values
df = df.fillna({'transactions': 0})

In [44]:
df.show(5)
df.count()

+----------+---------+--------+---+----------+-----------+------------+-----+----------+-------+-----------+----+-------+------------+----------+-------+
|      date|store_nbr|item_nbr| id|unit_sales|onpromotion|      family|class|perishable|   city|      state|type|cluster|transactions|dcoilwtico|holiday|
+----------+---------+--------+---+----------+-----------+------------+-----+----------+-------+-----------+----+-------+------------+----------+-------+
|2013-01-01|       25|  103665|  0|       7.0|      false|BREAD/BAKERY| 2712|         1|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  105574|  1|       1.0|      false|   GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  105575|  2|       2.0|      false|   GROCERY I| 1045|         0|Salinas|Santa Elena|   D|      1|         770|     93.14|   true|
|2013-01-01|       25|  108079|  3|       1.0|      false|   GROCERY I| 1030

127970257

In [45]:
output_path = f"{data_folder}/cleaned_data"
df.write.csv(output_path, header=True, mode="overwrite")

In [47]:
spark.stop()